In [1]:
!pip install kafka-python jaeger-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 888.1 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 2.9 MB/s eta 0:00:00a 0:00:01
  Created wheel for jaeger-client: filename=jaeger_client-4.8.0-py3-none-any.whl size=66029 sha256=cdfb8f496c923c12d3da9e2abada925a60d46a412b7045b6c2ad9347ce65f8ef
  Stored in directory: /home/jovyan/.cache/pip/wheels/44/c6/51/bd6a454e0a5f8ce568c515446aa30f5a4e6fbb575e34a1ca76
  Created wheel for opentracing: filename=opentracing-2.4.0-py3-none-any.whl size=51404 sha256=9f9e9f8eb64edd568d5d9923dd8e12176ecdcb1ba8f0478f280bc5987225fb8f
  Stored in directory: /home/jovyan/.cache/pip/wheels/d5/9a

In [2]:
import json
import random
import time
from datetime import datetime
from kafka import KafkaProducer
from jaeger_client import Config
import uuid

In [3]:
# Initialize Jaeger tracer
config = Config(
    config={
        'sampler': {'type': 'const', 'param': 1},
        'logging': True,
        'local_agent': {
            'reporting_host': 'jaeger',
            'reporting_port': 6831,
        },
    },
    service_name='producer',
    validate=True,
)
tracer = config.initialize_tracer()
print("Jaeger tracer initialized!")

Jaeger tracer initialized!


In [4]:
# Initialize Kafka producer
producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print("✅ Kafka producer connected!")

✅ Kafka producer connected!


In [5]:
# Event types and their characteristics
EVENT_TYPES = {
    'port_scan': {
        'ports': [22, 23, 80, 443, 3389, 8080],
        'protocols': ['TCP'],
        'suspicious': True,
        'mitre_tactic': 'Discovery',
        'mitre_technique': 'T1046'
    },
    'brute_force': {
        'ports': [22, 3389],
        'protocols': ['TCP'],
        'suspicious': True,
        'mitre_tactic': 'Credential Access',
        'mitre_technique': 'T1110'
    },
    'normal_web': {
        'ports': [80, 443],
        'protocols': ['TCP'],
        'suspicious': False,
        'mitre_tactic': None,
        'mitre_technique': None
    },
    'data_exfil': {
        'ports': [21, 22],
        'protocols': ['TCP', 'UDP'],
        'suspicious': True,
        'mitre_tactic': 'Exfiltration',
        'mitre_technique': 'T1048'
    },
    'dns_query': {
        'ports': [53],
        'protocols': ['UDP'],
        'suspicious': False,
        'mitre_tactic': None,
        'mitre_technique': None
    }
}

In [6]:
def generate_ip():
    """Generate random IP address"""
    return f"{random.randint(1, 255)}.{random.randint(0, 255)}.{random.randint(0, 255)}.{random.randint(1, 255)}"

def generate_event():
    """Generate a synthetic security event"""
    event_type = random.choice(list(EVENT_TYPES.keys()))
    config = EVENT_TYPES[event_type]
    
    event = {
        'event_id': str(uuid.uuid4()),
        'timestamp': datetime.utcnow().isoformat(),
        'event_type': event_type,
        'source_ip': generate_ip(),
        'dest_ip': generate_ip(),
        'source_port': random.randint(1024, 65535),
        'dest_port': random.choice(config['ports']),
        'protocol': random.choice(config['protocols']),
        'bytes_sent': random.randint(100, 100000),
        'bytes_received': random.randint(100, 100000),
    }
    
    return event

In [7]:
def produce_events(num_events=100, delay=0.1):
    """Generate and send events to Kafka"""
    
    print(f"Generating {num_events} events...\n")
    
    for i in range(num_events):
        with tracer.start_span('generate_event') as span:
            # Generate event
            event = generate_event()
            span.set_tag('event_id', event['event_id'])
            span.set_tag('event_type', event['event_type'])
            
            # Send to Kafka
            with tracer.start_span('kafka_send', child_of=span) as send_span:
                producer.send('events.raw', value=event)
                send_span.set_tag('topic', 'events.raw')
            
            if (i + 1) % 10 == 0:
                print(f"Sent {i + 1} events")
            
            time.sleep(delay)
    
    # Flush and close
    producer.flush()
    print(f"\nSuccessfully sent {num_events} events to Kafka!")
    print(f"View them at: http://localhost:8080")
    print(f"View traces at: http://localhost:16686")

In [8]:
produce_events(num_events=50, delay=0.1)

Generating 50 events...

Sent 10 events
Sent 20 events
Sent 30 events
Sent 40 events
Sent 50 events

Successfully sent 50 events to Kafka!
View them at: http://localhost:8080
View traces at: http://localhost:16686
